In [1]:
print("Hello, displacement project")

Hello, displacement project


In [2]:
import requests
import pandas as pd

url = "https://api.unhcr.org/population/v1/population/"
params = {
    "limit": 20,
    "year": 2022,
    "coa": "KEN",      # country of asylum = Kenya
    "cf_type": "ISO",  # use standard 3-letter country codes
}

response = requests.get(url, params=params, timeout=60)
print(response.status_code)

data = response.json()
df = pd.DataFrame(data["items"])
df

200


,year,coo_id,coo_name,coo,coo_iso,coa_id,coa_name,coa,coa_iso,refugees,asylum_seekers,returned_refugees,idps,returned_idps,stateless,ooc,oip,hst
0,2022,-,-,-,-,99,Kenya,KEN,KEN,504473,69011,3174,0,0,16779,87196,-,401356


## **1. Kenya refugee population over time**

**Objective:** Retrieve the number of refugees hosted in Kenya for each year from 2010 onward, and prepare the data for analysis.

**Source:** UNHCR Population Statistics API (`/population/`)

**Method:** Filter by country of asylum (`coa = KEN`) and a year range, then convert the count columns from text to numbers, since the API uses `-` for missing values.

In [3]:
# Request Kenya's data for every year from 2010 onward
params = {
    "limit": 100,
    "yearFrom": 2010,
    "yearTo": 2026,
    "coa": "KEN",
    "cf_type": "ISO",
}

response = requests.get(url, params=params, timeout=60)
kenya = pd.DataFrame(response.json()["items"])

# Keep only the columns needed
kenya = kenya[["year", "coa_name", "refugees", "asylum_seekers"]]

# Convert text to numbers; "-" becomes NaN (missing) instead of causing errors
for col in ["refugees", "asylum_seekers"]:
    kenya[col] = pd.to_numeric(kenya[col], errors="coerce")

kenya = kenya.sort_values("year")
kenya

,year,coa_name,refugees,asylum_seekers
0,2010,Kenya,402891,27957
1,2011,Kenya,566482,35268
2,2012,Kenya,564921,41940
3,2013,Kenya,534935,52271
4,2014,Kenya,551351,33994
5,2015,Kenya,553907,39953
6,2016,Kenya,451097,43748
7,2017,Kenya,431895,56499
8,2018,Kenya,421243,50458
9,2019,Kenya,438899,50829


## **2. Visualizing Kenya's refugee trend**

**Objective:** Plot refugees and asylum seekers hosted in Kenya from 2010 to 2026 to make the long-term trend visible.

**Method:** A line chart built with Plotly, with both series on the same axis so their scale can be compared.

**Note:** The latest year (2026) may reflect a mid-year count and should be interpreted with caution.

In [4]:
import plotly.express as px

# Reshape from "wide" (one column per category) to "long" (one row per
# year-category pair), which is the format Plotly expects for multiple lines
kenya_long = kenya.melt(
    id_vars="year",
    value_vars=["refugees", "asylum_seekers"],
    var_name="category",
    value_name="people",
)

fig = px.line(
    kenya_long,
    x="year",
    y="people",
    color="category",
    markers=True,
    title="Refugees and Asylum Seekers Hosted in Kenya, 2010 to 2026",
    labels={"year": "Year", "people": "Number of people", "category": "Category"},
)
fig.show()

### **Observations**

- Refugees hosted in Kenya rose from about 403,000 (2010) to a peak of about 566,000 (2011), fell to about 421,000 by 2018, then climbed steadily to about 631,000 in 2026.
- Asylum seekers stayed below roughly 70,000 until 2022, then rose sharply to about 220,000 by 2024 and have stayed near that level since.
- The 2026 figures may reflect a partial-year count and should be treated as provisional.

*Source: UNHCR Population Statistics API. Causes of these changes are not explained by this dataset.*

## **3. Refugees hosted across East Africa**

**Objective:** Compare the number of refugees hosted by East African countries from 2010 onward, to see which countries host the most and how this has changed.

**Method:** Loop over a list of country codes (ISO3), request each country's data from the UNHCR API, combine the results into one table, and convert the counts to numbers.

**Countries:** Kenya, Uganda, Ethiopia, Tanzania, South Sudan, Sudan, Somalia, Rwanda, Burundi, Djibouti, Eritrea. This list is a working definition of "East Africa" and can be adjusted.

In [5]:
# ISO3 codes for the countries we treat as "East Africa"
east_africa = {
    "KEN": "Kenya", "UGA": "Uganda", "ETH": "Ethiopia", "TZA": "Tanzania",
    "SSD": "South Sudan", "SDN": "Sudan", "SOM": "Somalia", "RWA": "Rwanda",
    "BDI": "Burundi", "DJI": "Djibouti", "ERI": "Eritrea",
}

frames = []
for code in east_africa:
    params = {
        "limit": 100,
        "yearFrom": 2010,
        "yearTo": 2026,
        "coa": code,
        "cf_type": "ISO",
    }
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()          # stop with a clear error if a request fails
    frames.append(pd.DataFrame(response.json()["items"]))

region = pd.concat(frames, ignore_index=True)
region = region[["year", "coa_name", "refugees", "asylum_seekers"]]

# Convert text to numbers; "-" becomes NaN (missing)
for col in ["refugees", "asylum_seekers"]:
    region[col] = pd.to_numeric(region[col], errors="coerce")

# Check: how many rows per country, and how many missing values?
print(region.groupby("coa_name").size())
print(region.isna().sum())


coa_name
Burundi                    17
Djibouti                   17
Eritrea                    17
Ethiopia                   17
Kenya                      17
Rwanda                     17
Somalia                    17
South Sudan                16
Sudan                      17
Uganda                     17
United Rep. of Tanzania    17
dtype: int64
year              0
coa_name          0
refugees          0
asylum_seekers    0
dtype: int64


## **4. Ranking East African host countries**

**Objective:** Identify which East African countries host the most refugees in the most recent year with a full-year count.

**Method:** Filter the regional table to a single year, sort by number of refugees, and plot a horizontal bar chart.

**Notes:**
- 2025 is used as the reference year because 2026 may be a partial-year count (to be verified against UNHCR's documentation).
- Raw counts do not account for country size; a population-adjusted comparison follows in a later section.
- Sudan's figures before 2011 include the territory that became South Sudan.

In [6]:
# Clean up a long country name
region["coa_name"] = region["coa_name"].replace(
    {"United Rep. of Tanzania": "Tanzania"}
)

# Choose the reference year (change this one value to compare other years)
ref_year = 2025

ranking = (
    region[region["year"] == ref_year]
    .sort_values("refugees", ascending=True)   # ascending so the largest bar ends up on top
)

fig = px.bar(
    ranking,
    x="refugees",
    y="coa_name",
    orientation="h",
    title=f"Refugees Hosted by East African Countries, {ref_year}",
    labels={"refugees": "Number of refugees", "coa_name": "Country"},
    text_auto=".2s",
)
fig.show()

### **Observations**

- Uganda hosts the most refugees in the region in 2025 (about 1.9 million), followed by Ethiopia (about 1.0 million) and Sudan (about 810,000).
- Kenya ranks fourth in absolute numbers (about 620,000), just ahead of South Sudan (about 600,000).
- Tanzania, Rwanda and Burundi host between about 110,000 and 170,000 each, while Djibouti, Somalia and Eritrea host very few.
- These are raw counts that do not account for population size, so they say who hosts the most people, not who carries the greatest relative burden.

## **5. Refugee trends by host country, 2010 to 2026**

**Objective:** Show how the number of refugees hosted has changed over time in each East African country.

**Method:** A grid of small line charts (one per country), ordered by the 2025 ranking. Each panel has its own vertical scale, so the **shape** of each country's trend is comparable, but the **size** of the numbers is not. Panel scales should be read individually.

**Note:** South Sudan has no 2010 record because it became independent in 2011.

In [7]:
# Order the panels from largest to smallest host (using the 2025 ranking)
panel_order = list(ranking["coa_name"][::-1])

fig = px.line(
    region,
    x="year",
    y="refugees",
    facet_col="coa_name",
    facet_col_wrap=4,                          # 4 panels per row
    category_orders={"coa_name": panel_order},
    height=700,
    title="Refugees Hosted by East African Countries, 2010 to 2026",
    labels={"year": "Year", "refugees": "Refugees"},
)

fig.update_yaxes(matches=None, showticklabels=True)   # each panel gets its own scale
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))  # "coa_name=Kenya" -> "Kenya"
fig.show()

### **Observations**

- **Uganda** shows the steepest long-term growth, from a few hundred thousand refugees in 2010 to about 1.9 million.
- **Ethiopia** and **South Sudan** also grew substantially over the period; South Sudan's rise is sharpest in the most recent years.
- **Kenya** rose to a peak in 2011, fell until 2018, then climbed steadily.
- **Tanzania** and **Rwanda** peaked in the mid-2010s and have declined since.
- **Burundi** has grown steadily, with a sharp rise in the last few years.
- **Eritrea** fell from several thousand refugees to near zero by the late 2010s.
- **Sudan** rose sharply in the 2010s, passed 1 million, and has since fallen back.
- Each panel uses its own vertical scale, so the shapes are comparable but the sizes are not.

## 6. Refugees relative to population

Objective: Compare host countries by the number of refugees per 1,000 inhabitants, to show relative burden rather than raw counts.

Method: Retrieve total population (World Bank indicator SP.POP.TOTL) for the same countries, merge it with the UNHCR table on country and year, and divide refugees by population.

Notes:
- The World Bank API can be slow, so requests are made one country at a time with automatic retries.
- World Bank population data usually lags behind UNHCR by a year or more, so the analysis uses the most recent year covered by both sources.

In [8]:
import time

def get_with_retry(url, params, tries=4, timeout=120):
    """Request a URL, retrying with a growing pause if it times out or fails."""
    for attempt in range(1, tries + 1):
        try:
            response = requests.get(url, params=params, timeout=timeout)
            response.raise_for_status()
            return response
        except requests.exceptions.RequestException as error:
            print(f"  attempt {attempt} failed: {type(error).__name__}")
            time.sleep(3 * attempt)
    raise RuntimeError(f"Request failed after {tries} attempts: {url}")

# 1. Get population from the World Bank API, one country at a time
population_frames = []
for code in east_africa:
    print("Fetching", code)
    wb_url = f"https://api.worldbank.org/v2/country/{code}/indicator/SP.POP.TOTL"
    wb_params = {"format": "json", "per_page": 100, "date": "2010:2026"}
    wb_response = get_with_retry(wb_url, wb_params)
    records = wb_response.json()[1]          # [0] is metadata, [1] is the data
    population_frames.append(pd.DataFrame({
        "iso3": [r["countryiso3code"] for r in records],
        "year": [int(r["date"]) for r in records],
        "population": [r["value"] for r in records],
    }))

population = pd.concat(population_frames, ignore_index=True)

# 2. Add ISO3 codes to our UNHCR table so the two tables can be joined
name_to_iso = {name: code for code, name in east_africa.items()}
region["iso3"] = region["coa_name"].map(name_to_iso)

# 3. Merge on country and year, then calculate the relative measure
merged = region.merge(population, on=["iso3", "year"], how="left")
merged["refugees_per_1000"] = merged["refugees"] / merged["population"] * 1000

# 4. Find the latest year that has both refugee and population data
latest_year = merged.dropna(subset=["population"])["year"].max()
print("Latest year with both sources:", latest_year)

burden = (
    merged[merged["year"] == latest_year]
    .sort_values("refugees_per_1000", ascending=True)
)

fig = px.bar(
    burden,
    x="refugees_per_1000",
    y="coa_name",
    orientation="h",
    title=f"Refugees per 1,000 Inhabitants, East Africa, {latest_year}",
    labels={"refugees_per_1000": "Refugees per 1,000 inhabitants", "coa_name": "Country"},
    text_auto=".1f",
)
fig.show()

Fetching KEN
Fetching UGA
Fetching ETH
Fetching TZA
Fetching SSD
Fetching SDN
Fetching SOM
Fetching RWA
Fetching BDI
Fetching DJI
Fetching ERI
Latest year with both sources: 2025


### **Observations**

- Measured per 1,000 inhabitants, South Sudan (49.4) and Uganda (37.7) carry the heaviest refugee burden in the region in 2025, followed by Djibouti (21.0) and Sudan (15.7).
- Kenya ranks fifth (10.7), close to Rwanda (7.9), Burundi (7.7) and Ethiopia (7.5).
- The ranking by population-adjusted burden differs from the ranking by raw count: Ethiopia drops from second to eighth, while Djibouti rises from ninth to third.

**Limitations**:
- This measure counts refugees hosted only. It does not include internally displaced people, who make up a large share of displacement in countries such as Sudan, South Sudan, Ethiopia and Somalia.
- Population figures are World Bank estimates for 2025 and may be revised.
- Refugee counts for the latest year may be provisional.

Sources: UNHCR Population Statistics API; World Bank (SP.POP.TOTL).

### **7. Ranking comparison and saving the processed data**

Objective: Show how each country's rank changes when refugee counts are adjusted for population, and save the final dataset for the project repository.

Method: Rank countries by raw count and by refugees per 1,000 inhabitants, then calculate the difference. A positive rank change means the country moves up once population size is taken into account.

In [11]:
# Compare the ranking by raw count with the ranking per 1,000 inhabitants
compare = merged[merged["year"] == latest_year][
    ["coa_name", "refugees", "population", "refugees_per_1000"]
].copy()

compare["rank_by_count"] = compare["refugees"].rank(ascending=False).astype(int)
compare["rank_per_1000"] = compare["refugees_per_1000"].rank(ascending=False).astype(int)
compare["rank_change"] = compare["rank_by_count"] - compare["rank_per_1000"]

compare = compare.sort_values("rank_per_1000")
display(compare.round(1))

# Save the processed dataset for the repository
processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)
merged.to_csv(processed_dir / "east_africa_refugees_population.csv", index=False)
print("Saved to data/processed/")

,coa_name,refugees,population,refugees_per_1000,rank_by_count,rank_per_1000,rank_change
82,South Sudan,601818,12188788.0,49.4,5,1,4
32,Uganda,1936719,51384894.0,37.7,1,2,-1
167,Djibouti,24899,1184076.0,21.0,9,3,6
99,Sudan,809816,51662147.0,15.7,3,4,-1
15,Kenya,617125,57532493.0,10.7,4,5,-1
133,Rwanda,115794,14569341.0,7.9,7,6,1
150,Burundi,110800,14390003.0,7.7,8,7,1
49,Ethiopia,1010466,135472051.0,7.5,2,8,-6
66,Tanzania,171142,70545865.0,2.4,6,9,-3
116,Somalia,20967,19654739.0,1.1,10,10,0


Saved to data/processed/


### **Observations**

- Adjusting for population changes the ranking substantially. Djibouti rises six places (from 9th by count to 3rd per 1,000 inhabitants) and South Sudan rises four (from 5th to 1st).
- Ethiopia hosts about 1.0 million refugees, the second largest number in the region, but ranks 8th per 1,000 inhabitants (7.5) because of its large population.
- Kenya ranks 4th by count and 5th per 1,000 inhabitants (10.7), a small shift.
- Somalia and Eritrea do not change rank. Eritrea's figure (119 refugees) is very small and should be checked against UNHCR's Refugee Data Finder before publication.